<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Humidity/Humidity_Statistical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load your datasets
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/humidity_training_dataset (1).csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/humidity_testing_dataset (1).csv")

target = "humidity"

# Optional: if you have datetime column
# train_df["last_updated"] = pd.to_datetime(train_df["last_updated"])
# test_df["last_updated"] = pd.to_datetime(test_df["last_updated"])

# train_df.set_index("last_updated", inplace=True)
# test_df.set_index("last_updated", inplace=True)

# Evaluation function
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(np.array(y_true) == 0, 1e-10, y_true)
    acc = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, acc

In [2]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load datasets
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/humidity_training_dataset (1).csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/humidity_testing_dataset (1).csv")


target = "humidity"

# Keep only target and make numeric
train_target = pd.to_numeric(train_df[target], errors="coerce").dropna()
test_target = pd.to_numeric(test_df[target], errors="coerce").dropna()

print("Train target length:", len(train_target))
print("Test target length:", len(test_target))

# Evaluation function
def evaluate(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(y_true == 0, 1e-10, y_true)
    accuracy = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, accuracy

# Fit ARIMA
model = ARIMA(train_target, order=(5, 1, 0))
model_fit = model.fit()

# Forecast exactly test length
forecast = model_fit.forecast(steps=len(test_target))

print("Forecast length:", len(forecast))

# Make sure both lengths match
min_len = min(len(test_target), len(forecast))
test_target = test_target.iloc[:min_len]
forecast = np.array(forecast)[:min_len]

metrics_arima = evaluate(test_target, forecast)

print("\nARIMA Results")
print("MSE:", metrics_arima[0])
print("RMSE:", metrics_arima[1])
print("MAE:", metrics_arima[2])
print("R2:", metrics_arima[3])
print("Accuracy (%):", metrics_arima[4])

Train target length: 104002
Test target length: 26001
Forecast length: 26001

ARIMA Results
MSE: 594.000612861255
RMSE: 24.37212778690558
MAE: 21.41940280613767
R2: -0.23909149950680697
Accuracy (%): 60.576937483697705


In [3]:
model = SARIMAX(
    train_df[target],
    order=(2,1,2),
    seasonal_order=(1,1,1,12)  # adjust if needed
)

model_fit = model.fit(disp=False)

forecast = model_fit.forecast(steps=len(test_df))

metrics_sarima = evaluate(test_df[target], forecast)

print("SARIMA Results")
print(metrics_sarima)

SARIMA Results
(1821.6162958187604, np.float64(42.680397090687435), 38.17407002621035, -2.7999106712021544, np.float64(47.85709108398047))


In [4]:
print("SARIMA Results")

print("MSE:", metrics_sarima[0])
print("RMSE:", metrics_sarima[1])
print("MAE:", metrics_sarima[2])
print("R2:", metrics_sarima[3])
print("Accuracy (%):", metrics_sarima[4])

SARIMA Results
MSE: 1821.6162958187604
RMSE: 42.680397090687435
MAE: 38.17407002621035
R2: -2.7999106712021544
Accuracy (%): 47.85709108398047


In [5]:
var_features = [
    "humidity",
    "temperature_celsius",
    "cloud",
    "precip_mm",
    "visibility_km",
    "air_quality_Ozone"
]

train_var = train_df[var_features].dropna()
test_var = test_df[var_features].dropna()

# =========================
# TRAIN VAR MODEL
# =========================
model = VAR(train_var)
model_fit = model.fit(maxlags=5)

lag_order = model_fit.k_ar

# =========================
# FORECAST
# =========================
forecast = model_fit.forecast(
    train_var.values[-lag_order:],
    steps=len(test_var)
)

forecast_df = pd.DataFrame(
    forecast,
    index=test_var.index,
    columns=var_features
)

# =========================
# EVALUATE (HUMIDITY)
# =========================
metrics_var = evaluate(
    test_var["humidity"],
    forecast_df["humidity"]
)

print("VAR Results (Humidity)")
print(metrics_var)

VAR Results (Humidity)
(560.7211530489766, np.float64(23.67955136925057), 20.612280556744974, -0.1696701977964592, np.float64(60.79816086455021))


In [6]:
results = pd.DataFrame([
    {
        "Model": "ARIMA",
        "MSE": metrics_arima[0],
        "RMSE": metrics_arima[1],
        "MAE": metrics_arima[2],
        "R2": metrics_arima[3],
        "Accuracy (%)": metrics_arima[4]
    },
    {
        "Model": "SARIMA",
        "MSE": metrics_sarima[0],
        "RMSE": metrics_sarima[1],
        "MAE": metrics_sarima[2],
        "R2": metrics_sarima[3],
        "Accuracy (%)": metrics_sarima[4]
    },
    {
        "Model": "VAR",
        "MSE": metrics_var[0],
        "RMSE": metrics_var[1],
        "MAE": metrics_var[2],
        "R2": metrics_var[3],
        "Accuracy (%)": metrics_var[4]
    }
]).round(4)

print(results)

    Model        MSE     RMSE      MAE      R2  Accuracy (%)
0   ARIMA   594.0006  24.3721  21.4194 -0.2391       60.5769
1  SARIMA  1821.6163  42.6804  38.1741 -2.7999       47.8571
2     VAR   560.7212  23.6796  20.6123 -0.1697       60.7982


In [7]:
import pandas as pd

stat_results_table = pd.DataFrame([
    {
        "Model": "ARIMA",
        "MSE": metrics_arima[0],
        "RMSE": metrics_arima[1],
        "MAE": metrics_arima[2],
        "R2": metrics_arima[3],
        "Accuracy (%)": metrics_arima[4]
    },
    {
        "Model": "SARIMA",
        "MSE": metrics_sarima[0],
        "RMSE": metrics_sarima[1],
        "MAE": metrics_sarima[2],
        "R2": metrics_sarima[3],
        "Accuracy (%)": metrics_sarima[4]
    },
    {
        "Model": "VAR",
        "MSE": metrics_var[0],
        "RMSE": metrics_var[1],
        "MAE": metrics_var[2],
        "R2": metrics_var[3],
        "Accuracy (%)": metrics_var[4]
    }
]).round(4)

print(stat_results_table)

    Model        MSE     RMSE      MAE      R2  Accuracy (%)
0   ARIMA   594.0006  24.3721  21.4194 -0.2391       60.5769
1  SARIMA  1821.6163  42.6804  38.1741 -2.7999       47.8571
2     VAR   560.7212  23.6796  20.6123 -0.1697       60.7982


In [8]:
stat_results_table.to_csv("statistical_model_results.csv", index=False)

print("Statistical results saved as CSV!")

Statistical results saved as CSV!


In [9]:
from google.colab import files
files.download("statistical_model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>